# Tratando erros

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Capturar uma exceção com try e catch, e manter o programa rodando.
- Lançar uma exceção quando um método recebe um valor que não pode aceitar.
- Criar sua própria classe de exceção.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. O que é uma exceção
2. try e catch: tratando a exceção
3. Os suspeitos de sempre
4. Vários catch, e finally
5. throw: recusando um valor ruim
6. Exceções sobem pela pilha de chamadas
7. A sua própria classe de exceção

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## 1. O que é uma exceção

Quando algo dá errado enquanto o programa roda (uma divisão por zero, um índice depois do fim, um texto que não é número), o Java **lança uma exceção**: ele para o que estava fazendo e, a não ser que algum código a trate, o programa inteiro para com uma mensagem. Essa mensagem diz o **tipo** da exceção, um **detalhe** e a **linha** onde aconteceu. Lê-la é o primeiro passo para consertar o bug.

In [ ]:
int[] scores = {7, 9, 4};
int index = 2;
IO.println("Antes");
IO.println(scores[index]);
IO.println("Depois");

🧪 **Experimente:** mude `index` para `3` e rode: ache o tipo da exceção, o detalhe (qual índice, qual tamanho) e a linha. O "Depois" é impresso?

## 2. try e catch: tratando a exceção

Ponha o código arriscado num bloco `try`. Se ele lançar, o Java pula direto para o `catch` correspondente, e o programa **continua rodando** depois dele. Se nada for lançado, o `catch` é pulado. `e.getMessage()` devolve o detalhe.

```java
try {
    // código que pode lançar
} catch (NumberFormatException e) {
    // roda só se esse tipo foi lançado
}
```

In [ ]:
String[] typed = {"12", "abc", "7"};
for (String text : typed) {
    try {
        int n = Integer.parseInt(text);
        IO.println(text + " em dobro é " + n * 2);
    } catch (NumberFormatException e) {
        IO.println("\"" + text + "\" não é número: " + e.getMessage());
    }
}
IO.println("O programa continua");

🧪 **Experimente:** acrescente `"4.5"` e `""` ao array. Depois mova o `try` para fora do laço: por que agora o programa para no primeiro valor ruim?

## 3. Os suspeitos de sempre

| Exceção | Costuma significar |
|---|---|
| `NumberFormatException` | `parseInt` / `parseDouble` recebeu um texto que não é número |
| `ArithmeticException` | uma divisão inteira por zero |
| `ArrayIndexOutOfBoundsException` | um índice abaixo de 0 ou de pelo menos `length` |
| `NullPointerException` | usar uma variável que guarda `null` como se apontasse para um objeto |
| `IllegalArgumentException` | um método recebeu um valor que não aceita |

In [ ]:
try {
    IO.println(10 / 0);
} catch (ArithmeticException e) {
    IO.println("ArithmeticException: " + e.getMessage());
}
try {
    String name = null;
    IO.println(name.length());
} catch (NullPointerException e) {
    IO.println("NullPointerException: a variável guardava null");
}
IO.println(10 / 0.0);   // com double não há exceção: Infinity

🧪 **Experimente:** mude o primeiro `catch` para `NumberFormatException`: a exceção não combina mais, então ninguém a captura e a célula para. Volte como estava.

### ✏️ Exercício: Um índice seguro

Devolva values[i], ou -1 quando i estiver fora do array. Capture a ArrayIndexOutOfBoundsException em vez de conferir antes.

In [ ]:
int safeGet(int[] values, int i) {
    // TODO: leia values[i] dentro do try
    return -1;
}

void main() {
    check("dentro", safeGet(new int[] {4, 5}, 1), 5);
    check("longe demais", safeGet(new int[] {4, 5}, 2), -1);
    check("negativo", safeGet(new int[] {4, 5}, -1), -1);
}

<details>
<summary>Ver uma resposta</summary>

```java
int safeGet(int[] values, int i) {
    try {
        return values[i];
    } catch (ArrayIndexOutOfBoundsException e) {
        return -1;
    }
}

void main() {
    check("dentro", safeGet(new int[] {4, 5}, 1), 5);
    check("longe demais", safeGet(new int[] {4, 5}, 2), -1);
    check("negativo", safeGet(new int[] {4, 5}, -1), -1);
}
```

</details>

## 4. Vários catch, e finally

Um `try` pode ter vários `catch`, um por tipo; o Java usa o primeiro que combina, então ponha os tipos **mais específicos** primeiro. Um bloco `finally` roda **sempre**, tendo havido exceção ou não, o que serve para trabalho de limpeza, como fechar um arquivo.

In [ ]:
int[] values = {10, 0, 5};
String[] picks = {"0", "1", "7", "x"};
for (String pick : picks) {
    try {
        int i = Integer.parseInt(pick);
        IO.println("100 / values[" + i + "] = " + 100 / values[i]);
    } catch (NumberFormatException e) {
        IO.println(pick + ": não é número");
    } catch (ArithmeticException e) {
        IO.println(pick + ": divisão por zero");
    } catch (ArrayIndexOutOfBoundsException e) {
        IO.println(pick + ": esse índice não existe");
    } finally {
        IO.println("  (terminou " + pick + ")");
    }
}

🧪 **Experimente:** acrescente um `catch (IllegalArgumentException e)` antes do de `NumberFormatException` e leia o erro: `NumberFormatException` é um tipo de `IllegalArgumentException`, então o segundo catch nunca poderia rodar.

## 5. throw: recusando um valor ruim

Os seus métodos também podem lançar. Quando um método recebe um valor com que não consegue trabalhar, `throw new IllegalArgumentException("motivo")` o interrompe ali mesmo, em vez de devolver uma resposta errada. Quem chamou decide o que fazer, com um `try`.

In [ ]:
double average(int[] values) {
    if (values.length == 0) {
        throw new IllegalArgumentException("nenhum valor para a média");
    }
    int total = 0;
    for (int v : values) {
        total += v;
    }
    return (double) total / values.length;
}

void main() {
    IO.println(average(new int[] {7, 8, 9}));
    try {
        IO.println(average(new int[] {}));
    } catch (IllegalArgumentException e) {
        IO.println("Não deu para calcular: " + e.getMessage());
    }
}

🧪 **Experimente:** tire o `if` com o `throw` e rode: o que a média de nada dá agora? O que é melhor, uma exceção ou um número estranho?

### ✏️ Exercício: Lance em entrada inválida

Devolva a idade quando ela for 0 ou mais; senão lance `new IllegalArgumentException("negative age")`. O teste a captura e lê a mensagem.

In [ ]:
/** Calls checkedAge and returns the exception's message, or "no exception". */
String messageFor(int age) {
    try {
        checkedAge(age);
        return "no exception";
    } catch (IllegalArgumentException e) {
        return e.getMessage();
    }
}

int checkedAge(int age) {
    // TODO: lance quando age < 0
    return age;
}

void main() {
    check("uma idade válida", checkedAge(30), 30);
    check("zero", checkedAge(0), 0);
    check("negativa", messageFor(-1), "negative age");
}

<details>
<summary>Ver uma resposta</summary>

```java
/** Calls checkedAge and returns the exception's message, or "no exception". */
String messageFor(int age) {
    try {
        checkedAge(age);
        return "no exception";
    } catch (IllegalArgumentException e) {
        return e.getMessage();
    }
}

int checkedAge(int age) {
    if (age < 0) {
        throw new IllegalArgumentException("negative age");
    }
    return age;
}

void main() {
    check("uma idade válida", checkedAge(30), 30);
    check("zero", checkedAge(0), 0);
    check("negativa", messageFor(-1), "negative age");
}
```

</details>

## 6. Exceções sobem pela pilha de chamadas

Uma exceção que não é capturada onde foi lançada sai daquele método e vai para quem o chamou, depois para quem chamou esse, e assim por diante, até algum `catch` tratá-la, ou ela sair do `main` e parar o programa. Os slides acompanham uma exceção subindo pela pilha de chamadas.

#### 🎞️ Uma exceção subindo pela pilha de chamadas

![parseInt lança uma NumberFormatException; ela sai de cada método sem catch até o main capturá-la](exception-steps.pt-br.svg)

In [ ]:
int age(String text) {
    IO.println("  age(\"" + text + "\") começa");
    int value = Integer.parseInt(text);     // pode lançar aqui
    IO.println("  age termina");            // pulado quando lança
    return value;
}

void main() {
    for (String text : new String[] {"30", "x"}) {
        try {
            IO.println("Idade: " + age(text));
        } catch (NumberFormatException e) {
            IO.println("o main capturou: " + e.getMessage());
        }
    }
}

🧪 **Experimente:** mova o `try` / `catch` para dentro de `age`, devolvendo -1 quando falhar: agora o main nunca vê a exceção. Qual lugar faz mais sentido?

## 7. A sua própria classe de exceção

Uma exceção é um objeto de uma classe, então você pode criar a sua para problemas específicos do seu programa. Estenda `RuntimeException`, e quem chama não é obrigado a capturá-la. Estenda `Exception`, e ela passa a ser **verificada**: todo método que pode lançá-la precisa dizer `throws` no cabeçalho, e o compilador obriga quem chama a tratá-la.

In [ ]:
class EmptyNameException extends Exception {
    EmptyNameException() {
        super("o nome está vazio");
    }
}

String greeting(String name) throws EmptyNameException {
    if (name.isBlank()) {
        throw new EmptyNameException();
    }
    return "Oi, " + name.trim() + "!";
}

void main() {
    for (String name : new String[] {"Ada", "   "}) {
        try {
            IO.println(greeting(name));
        } catch (EmptyNameException e) {
            IO.println("Erro: " + e.getMessage());
        }
    }
}

🧪 **Experimente:** tire o `try` / `catch` do `main` e leia o erro: uma exceção verificada precisa ser capturada ou declarada. Depois troque `extends Exception` por `extends RuntimeException`: compila agora?

## Em resumo

Quando algo dá errado, o Java **lança** uma exceção, e o programa para, a menos que algum código a **capture**. Coloque o código arriscado no `try`; um bloco `catch` só roda quando o tipo de exceção dele é lançado.

```java
int age(String text) {
    int value = Integer.parseInt(text.trim());
    if (value < 0 || value > 150) {
        throw new IllegalArgumentException("Não é uma idade real: " + value);
    }
    return value;
}

void main() {
    String line = IO.readln();
    try {
        IO.println("Ano que vem: " + (age(line) + 1));
    } catch (NumberFormatException e) {
        IO.println("Não é um número: " + line);
    } catch (IllegalArgumentException e) {
        IO.println(e.getMessage());
    }
}
```

Sua própria exceção é uma classe que estende `Exception` (quem chama tem que capturá-la ou declarar `throws`) ou `RuntimeException` (não precisa).

**Cuidado:** coloque o `catch` mais específico primeiro. `NumberFormatException` é um tipo de `IllegalArgumentException`, então na outra ordem o compilador recusa o código.

## Leituras

- [Catching and Handling Exceptions](https://dev.java/learn/language/annotations-exceptions/exceptions/catching-handling/) (dev.java, 15 min): Blocos try e catch, capturar mais de uma exceção, e finally. O exemplo principal escreve num arquivo; se parecer pesado, a página sugere pular para os padrões comuns no fim. Try-with-resources é mais do que você precisa agora.
- [Throwing Exceptions](https://dev.java/learn/language/annotations-exceptions/exceptions/throwing/) (dev.java, 8 min): Opcional, para Saldo insuficiente: o comando throw, a cláusula throws, e criar sua própria classe de exceção. Exceções encadeadas são mais do que você precisa agora.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Divisão segura (try / catch)** · Fácil
- **Ler números (NumberFormatException)** · Médio
- **Pergunte até ser válido** · Médio
- **Saldo insuficiente (exceção própria)** · Médio
- **Calculadora robusta** · Difícil